In [0]:
%sql
-----  Creating new catalog, schema  ------ 
create catalog if not exists sql_youtube_practise;
use catalog sql_youtube_practise;
create schema if not exists sql;
use sql;
show current schema;

catalog,namespace
sql_youtube_practise,sql


#### Question1 - 
- ##### We have a table which stores data of multiple sections. 
- ##### Every section has 3 numbers we have to find top 4 numbers from any 2 sections(2 numbers each) whose addition should be maximum. 
- ##### So in this case we will choose section b where we have 19(10+9) then we need to choose either C or D because both has sum of 18 but in D we have 10 which is big from 9 so we will give priority to D.

In [0]:
%sql
DROP TABLE IF EXISTS section_data;
CREATE TABLE section_data (
    section VARCHAR(5),
    number INT
);
INSERT INTO section_data VALUES
('A', 5),
('A', 7),
('A', 10),
('B', 7),
('B', 9),
('B', 10),
('C', 9),
('C', 7),
('C', 9),
('D', 10),
('D', 3),
('D', 8);
SELECT * FROM section_data;

section,number
A,5
A,7
A,10
B,7
B,9
B,10
C,9
C,7
C,9
D,10


In [0]:
%sql
with cte as (
    select *, 
    row_number() over(partition by section order by number desc) as rnk
    from section_data
),
cte1 as (
    select *, 
        sum(number) over(partition by section) as sum,
        max(number) over(partition by section) as max_per_section
    from cte where rnk <=2
),
cte2 as (
    select *, 
        dense_rank() over(order by sum desc, max_per_section desc) as dnk
    from cte1
)
select * from cte2 
where dnk <=2 
order by dnk, section;

section,number,rnk,sum,max_per_section,dnk
B,10,1,19,10,1
B,9,2,19,10,1
D,10,1,18,10,2
D,8,2,18,10,2


#### Question2:
##### SQL Interview Questions - Booking Analysis --> Series of questions are attached below:

In [0]:
%sql
DROP TABLE IF EXISTS booking_table;
DROP TABLE IF EXISTS user_table;
CREATE TABLE booking_table (
    Booking_id       VARCHAR(3) NOT NULL,
    Booking_date     DATE NOT NULL,
    User_id          VARCHAR(3) NOT NULL,
    Line_of_business VARCHAR(6) NOT NULL
);
INSERT INTO booking_table (Booking_id, Booking_date, User_id, Line_of_business) VALUES
('b1','2022-03-23','u1','Flight'),
('b2','2022-03-27','u2','Flight'),
('b3','2022-03-28','u1','Hotel'),
('b4','2022-03-31','u4','Flight'),
('b5','2022-04-02','u1','Hotel'),
('b6','2022-04-02','u2','Flight'),
('b7','2022-04-06','u5','Flight'),
('b8','2022-04-06','u6','Hotel'),
('b9','2022-04-06','u2','Flight'),
('b10','2022-04-10','u1','Flight'),
('b11','2022-04-12','u4','Flight'),
('b12','2022-04-16','u1','Flight'),
('b13','2022-04-19','u2','Flight'),
('b14','2022-04-20','u5','Hotel'),
('b15','2022-04-22','u6','Flight'),
('b16','2022-04-26','u4','Hotel'),
('b17','2022-04-28','u2','Hotel'),
('b18','2022-04-30','u1','Hotel'),
('b19','2022-05-04','u4','Hotel'),
('b20','2022-05-06','u1','Flight');

CREATE TABLE user_table (
    User_id VARCHAR(3) NOT NULL,
    Segment VARCHAR(2) NOT NULL
);
INSERT INTO user_table (User_id, Segment) VALUES
('u1','s1'),
('u2','s1'),
('u3','s1'),
('u4','s2'),
('u5','s2'),
('u6','s3'),
('u7','s3'),
('u8','s3'),
('u9','s3'),
('u10','s3');
SELECT * FROM booking_table;

Booking_id,Booking_date,User_id,Line_of_business
b1,2022-03-23,u1,Flight
b2,2022-03-27,u2,Flight
b3,2022-03-28,u1,Hotel
b4,2022-03-31,u4,Flight
b5,2022-04-02,u1,Hotel
b6,2022-04-02,u2,Flight
b7,2022-04-06,u5,Flight
b8,2022-04-06,u6,Hotel
b9,2022-04-06,u2,Flight
b10,2022-04-10,u1,Flight


In [0]:
%sql
SELECT * FROM user_table;

User_id,Segment
u1,s1
u2,s1
u3,s1
u4,s2
u5,s2
u6,s3
u7,s3
u8,s3
u9,s3
u10,s3


##### Q1. Segment-Level Booking Analysis
Write an SQL query that gives the below output:
| Segment | Total_user_count | User_who_booked_flight_in_apr2022 |
|--------|------------------:|-----------------------------------:|
| s1     | 3                 | 2                                  |
| s2     | 2                 | 2                                  |
| s3     | 5                 | 1                                  |

In [0]:
%sql
------------------------------------ Approach 1 -------------------------------------------

with cte as (
    select * from booking_table 
    where Line_of_business = 'Flight' and Booking_date between '2022-04-01' and '2022-04-30'
),
cte1 as (
    select Segment, count(User_id) as count
    from user_table
    group by Segment
),
cte2 as (
    select count(distinct cte.User_id) as count_users, u.Segment from cte
    join user_table u
    on cte.User_id = u.User_id
    group by u.Segment
)
select * from cte1
join cte2
on cte1.Segment = cte2.Segment;

Segment,count,count_users,Segment
s1,3,2,s1
s2,2,2,s2
s3,5,1,s3


In [0]:
%sql
------------------------------------------- Approach 2 ---------------------------------------------

select u.Segment, count(distinct u.User_id) as no_of_users,
count(distinct
    case
        when b.Line_of_business = 'Flight' and b.Booking_date between '2022-04-01' and '2022-04-30' then u.User_id
    end
) as Users_who_booked_flight_on_Apr2022
from user_table u
left join booking_table b
on u.User_id = b.User_id
group by u.Segment;

Segment,no_of_users,Users_who_booked_flight_on_Apr2022
s2,2,2
s1,3,2
s3,5,1


##### Q2. First Booking Was a Hotel

Write a query to identify users whose **first booking was a hotel booking**.

In [0]:
%sql
with cte as (
select *,
row_number() over(partition by User_id order by Booking_date) as rnk
from booking_table
)
select * from cte
where rnk = 1 and Line_of_business = 'Hotel';

Booking_id,Booking_date,User_id,Line_of_business,rnk
b8,2022-04-06,u6,Hotel,1


##### Q3. Days Between First and Last Booking

Write a query to calculate the **number of days between the first and last booking of each user**.

In [0]:
%sql
select User_id, min(Booking_date), max(Booking_date), date_diff(day, min(Booking_date), max(Booking_date)) as diff
from booking_table
group by User_id

User_id,min(Booking_date),max(Booking_date),diff
u1,2022-03-23,2022-05-06,44
u2,2022-03-27,2022-04-28,32
u4,2022-03-31,2022-05-04,34
u5,2022-04-06,2022-04-20,14
u6,2022-04-06,2022-04-22,16


##### Q4. Flight and Hotel Bookings by Segment

Write a query to count the **number of flight and hotel bookings for each user segment for the year 2022**.

In [0]:
%sql
with cte as (
    select u.Segment, b.Line_of_business, count(Line_of_business) as Count from user_table as u
    join booking_table as b
    on u.User_id = b.User_id
    group by u.Segment, b.Line_of_business
)
select
    Segment,
    sum(case
        when Line_of_business = "Flight" then Count
    end) as Flight_Bookings,
    sum(case
        when Line_of_business = "Hotel" then Count
    end) as Hotel_Bookings
from cte
group by Segment;

Segment,Flight_Bookings,Hotel_Bookings
s1,8,4
s2,3,3
s3,1,1


##### Question3: Merge overlapping events in the same hall

In [0]:
%sql
DROP TABLE IF EXISTS hall_events;
CREATE TABLE hall_events (
    hall_id INT,
    start_date DATE,
    end_date DATE
);
INSERT INTO hall_events VALUES
(1, '2023-01-13', '2023-01-14'),
(1, '2023-01-14', '2023-01-17'),
(1, '2023-01-15', '2023-01-17'),
(1, '2023-01-18', '2023-01-25'),
(2, '2022-12-09', '2022-12-23'),
(2, '2022-12-13', '2022-12-17'),
(3, '2022-12-01', '2023-01-30');

SELECT * FROM hall_events ORDER BY hall_id, start_date;

hall_id,start_date,end_date
1,2023-01-13,2023-01-14
1,2023-01-14,2023-01-17
1,2023-01-15,2023-01-17
1,2023-01-18,2023-01-25
2,2022-12-09,2022-12-23
2,2022-12-13,2022-12-17
3,2022-12-01,2023-01-30


In [0]:
%sql
with cte as (
    select *,
        lag(end_date) over (partition by hall_id order by start_date) as prev_end_date
        from hall_events
),
cte1 as (
    select *,
        sum(
            case 
                when prev_end_date is null or start_date > prev_end_date 
                then 1 else 0 
            end
        ) over (partition by hall_id order by start_date) as grp
    from cte
)
select hall_id, min(start_date) as start_date, max(end_date) as end_date
from cte1
group by hall_id, grp

hall_id,start_date,end_date
1,2023-01-13,2023-01-17
1,2023-01-18,2023-01-25
2,2022-12-09,2022-12-23
3,2022-12-01,2023-01-30
